# 04 - Data Cleaning (Exploratory Only)

**IMPORTANT:** This notebook is **exploratory only**.

---

**Input:** `data/interim/train.csv` | `val.csv` | `test.csv`

**Output:** `data/processed/train_clean.csv` | `val_clean.csv` | `test_clean.csv`

**Artifacts:** `artifacts/cleaning_artifacts.json` | `lof_model.pkl`

**Config:** `configs/data_config.yaml` -> `eda_derived` section

---

### What this notebook does (Exploratory)

| Step | Action | Source / Note |
| :--- | :--- | :--- |
| 0 | Setup environment, imports, and `PROJECT_DIR` | `src.utils.paths`; `touch __init__.py` removed |
| 1 | Verify interim files exist locally | No DVC pull (handled in `00_environment_setup`) |
| 2 | Load train/val/test from interim with empty checks | `DataLoader` + `if df is None or df.empty` |
| 3 | Load EDA-derived config from YAML | `data_config.yaml` -> `eda_derived` (no `source` or `log1p` fields) |
| 4 | Run cleaning pipeline (imputation + LOF) | `run_cleaning()` with `auto_track_dvc=False` |
| 5 | Verify cleaning results (nulls, `lof_outlier`, shapes) | Exploratory validation only |
| 6 | Check saved artifacts (JSON + PKL) | `artifacts/cleaning_artifacts.json` & `lof_model.pkl` |

**Note:** `is_capped` flag and `log1p` transformations are **not** applied here. They belong to feature engineering (`05_feature_engineering.ipynb` or `dvc repro` stages).

### FIT/TRANSFORM rule

All statistics (median, LOF model) are **fit on train only** and applied to val/test.
This prevents data leakage from test information into training decisions.

---

---
## 0 - Setup & Import

In [35]:
# ===================================================================
# Section 0 · Setup
# ===================================================================
import os
import sys
from pathlib import Path

from src.utils.paths import PROJECT_DIR

os.chdir(PROJECT_DIR)
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

import importlib
importlib.invalidate_caches()

print(f"✅ Working dir : {os.getcwd()}")
print(f"✅ sys.path[0] : {sys.path[0]}")

✅ Working dir : /content/california_housing_full_project
✅ sys.path[0] : /content/california_housing_full_project


In [36]:
import logging
import pandas as pd
import numpy as np

from src.utils.logger import setup_logging, get_logger
from src.data.data_loader import DataLoader
from src.data.cleaning import (
    run_cleaning,
    load_eda_config,
    CleaningResult,
    CleaningError,
)

setup_logging(level=logging.INFO)
logger = get_logger("notebook.04_cleaning")

CONFIG_PATH = "configs/data_config.yaml"

print("Imports ready")

Imports ready


---
## 1 - Load Interim Splits

In [37]:
loader = DataLoader()

train = loader.load_interim("train.csv")
val = loader.load_interim("val.csv")
test = loader.load_interim("test.csv")

if train is None or train.empty:
    raise ValueError("train.csv is empty or None")
if val is None or val.empty:
    raise ValueError("val.csv is empty or None")
if test is None or test.empty:
    raise ValueError("test.csv is empty or None")

print(f"train : {train.shape[0]:,} rows x {train.shape[1]} cols")
print(f"val   : {val.shape[0]:,} rows x {val.shape[1]} cols")
print(f"test  : {test.shape[0]:,} rows x {test.shape[1]} cols")
print()
print("Null counts (train):")
nulls = train.isnull().sum()
print(nulls[nulls > 0].to_string() if nulls.sum() > 0 else "  No nulls")

2026-09-02 23:29:28 | INFO     | src.data.data_loader | DataLoader initialized | Drive mode: True
2026-09-02 23:29:28 | INFO     | src.data.data_loader | Loading: /content/california_housing_full_project/data/interim/train.csv


2026-09-02 23:29:28 | INFO     | src.data.data_loader | Loaded 'train.csv' | shape=(14448, 10) | stage=interim
2026-09-02 23:29:28 | INFO     | src.data.data_loader | Loading: /content/california_housing_full_project/data/interim/val.csv
2026-09-02 23:29:28 | INFO     | src.data.data_loader | Loaded 'val.csv' | shape=(3096, 10) | stage=interim
2026-09-02 23:29:28 | INFO     | src.data.data_loader | Loading: /content/california_housing_full_project/data/interim/test.csv
2026-09-02 23:29:28 | INFO     | src.data.data_loader | Loaded 'test.csv' | shape=(3096, 10) | stage=interim
train : 14,448 rows x 10 cols
val   : 3,096 rows x 10 cols
test  : 3,096 rows x 10 cols

Null counts (train):
total_bedrooms    140


---
## 2 - Verify EDA Config

Before cleaning, confirm that `data_config.yaml` has the `eda_derived` section
with real values from `notebooks/03_eda.ipynb`. If `source = fallback`, stop and
update the config first.

In [40]:
eda_cfg = load_eda_config(CONFIG_PATH)

print(f"impute columns       : {eda_cfg.impute_columns}")
print(f"imputation strategy  : {eda_cfg.imputation_strategy}")
print(f"cap threshold        : ${eda_cfg.cap_threshold:,.0f}")
print(f"LOF contamination    : {eda_cfg.lof_contamination}")
print(f"LOF n_neighbors      : {eda_cfg.lof_n_neighbors}")
print(f"LOF features         : {eda_cfg.lof_features}")
print()
print("Config loaded successfully - ready to clean")

2026-09-02 23:31:25 | INFO     | src.data.cleaning | EDA configuration loaded successfully.
2026-09-02 23:31:25 | INFO     | src.data.cleaning | Imputation columns: ['total_bedrooms']
2026-09-02 23:31:25 | INFO     | src.data.cleaning | LOF features: ['median_income', 'total_rooms', 'population', 'households', 'longitude', 'latitude']


2026-09-02 23:31:25 | INFO     | src.data.cleaning | LOF contamination: 0.02
2026-09-02 23:31:25 | INFO     | src.data.cleaning | LOF n_neighbors: 20
impute columns       : ['total_bedrooms']
imputation strategy  : {'total_bedrooms': 'median'}
cap threshold        : $500,001
LOF contamination    : 0.02
LOF n_neighbors      : 20
LOF features         : ['median_income', 'total_rooms', 'population', 'households', 'longitude', 'latitude']

Config loaded successfully - ready to clean


## 3 · Run Cleaning Pipeline

`run_cleaning()` applies all steps in order:

1. **Fit imputer on train** → apply to all three splits (`total_bedrooms` median)
2. **Fit LOF on train** → apply outlier flag to all three splits (`lof_outlier` column)
3. **Save to `data/processed/`**

> 🚫 `is_capped` flag and `log1p` transformations are **NOT applied** here.
> They belong to feature engineering (`05_feature_engineering.ipynb` or `dvc repro`).

In [41]:
try:
    result = run_cleaning(
        train=train,
        val=val,
        test=test,
        config_path=CONFIG_PATH,
        save_artifacts_flag=True,
    )
    print(result.summary())
except CleaningError as e:
    logger.error(f"Cleaning pipeline failed: {e}")
    print(f"ERROR: Cleaning failed - {e}")
    raise
except Exception as e:
    logger.error(f"Unexpected error during cleaning: {e}")
    print(f"UNEXPECTED ERROR: {e}")
    raise

2026-09-02 23:32:53 | INFO     | src.data.cleaning | ======================================================================
2026-09-02 23:32:53 | INFO     | src.data.cleaning | DATA CLEANING STARTED
2026-09-02 23:32:53 | INFO     | src.data.cleaning | ======================================================================
2026-09-02 23:32:53 | INFO     | src.data.cleaning | Step 1/6 - Validating cleaning inputs...
2026-09-02 23:32:53 | INFO     | src.data.cleaning | Step 2/6 - Loading EDA-derived configuration...
2026-09-02 23:32:53 | INFO     | src.data.cleaning | EDA configuration loaded successfully.
2026-09-02 23:32:53 | INFO     | src.data.cleaning | Imputation columns: ['total_bedrooms']
2026-09-02 23:32:53 | INFO     | src.data.cleaning | LOF features: ['median_income', 'total_rooms', 'population', 'households', 'longitude', 'latitude']
2026-09-02 23:32:53 | INFO     | src.data.cleaning | LOF contamination: 0.02
2026-09-02 23:32:53 | INFO     | src.data.cleaning | LOF n_neighbors

## 4 · Verify Cleaning Results

Three checks:
- ✅ **Nulls removed** — `total_bedrooms` should have zero nulls
- ✅ **LOF flag added** — `lof_outlier` column must exist in all splits
- ✅ **median_income unchanged** — NOT log-transformed (tree models)

> 🚫 We do **NOT** check for `is_capped` or `log1p` — these are deferred
> to feature engineering (`05_feature_engineering.ipynb`).

In [42]:
print("-- Null check after cleaning --")
for name, df in [("train", result.train), ("val", result.val), ("test", result.test)]:
    nulls = df.isnull().sum().sum()
    status = "OK" if nulls == 0 else f"FAIL ({nulls} nulls remaining)"
    print(f"  {name:<6} : {status}")

-- Null check after cleaning --
  train  : OK
  val    : OK
  test   : OK


In [44]:
print("-- New columns check --")
for col in ["lof_outlier"]:
    for name, df in [("train", result.train), ("val", result.val), ("test", result.test)]:
        present = col in df.columns
        print(f"  {col:<15} in {name:<6} : {'OK' if present else 'MISSING'}")

print()
print("-- lof_outlier distribution --")
for name, df in [("train", result.train), ("val", result.val), ("test", result.test)]:
    counts = df["lof_outlier"].value_counts().sort_index()
    print(f"  {name:<6} : {dict(counts)} (-99=unknown, 0=inlier, 1=outlier)")

-- New columns check --
  lof_outlier     in train  : OK
  lof_outlier     in val    : OK
  lof_outlier     in test   : OK

-- lof_outlier distribution --
  train  : {0: np.int64(14208), 1: np.int64(240)} (-99=unknown, 0=inlier, 1=outlier)
  val    : {0: np.int64(3047), 1: np.int64(49)} (-99=unknown, 0=inlier, 1=outlier)
  test   : {0: np.int64(3034), 1: np.int64(62)} (-99=unknown, 0=inlier, 1=outlier)


In [46]:
print("-- Null check after cleaning --")
for name, df in [("train", result.train), ("val", result.val), ("test", result.test)]:
    nulls = df.isnull().sum().sum()
    status = "OK" if nulls == 0 else f"FAIL ({nulls} nulls remaining)"
    print(f"  {name:<6} : {status}")

print()
print("-- New columns check --")
for col in ["lof_outlier"]:
    for name, df in [("train", result.train), ("val", result.val), ("test", result.test)]:
        present = col in df.columns
        print(f"  {col:<15} in {name:<6} : {'OK' if present else 'MISSING'}")

print()
print("-- lof_outlier distribution --")
for name, df in [("train", result.train), ("val", result.val), ("test", result.test)]:
    counts = df["lof_outlier"].value_counts().sort_index()
    print(f"  {name:<6} : {dict(counts)} (-99=unknown, 0=inlier, 1=outlier)")

print()
print("-- Shape check --")
for name, df in [("train", result.train), ("val", result.val), ("test", result.test)]:
    print(f"  {name:<6} : {df.shape[0]:,} rows x {df.shape[1]} cols")

print()
print("-- Column consistency --")
train_cols = list(result.train.columns)
for name, df in [("val", result.val), ("test", result.test)]:
    if list(df.columns) == train_cols:
        print(f"  {name} columns match train")
    else:
        diff = set(train_cols) ^ set(df.columns)
        print(f"  {name} column mismatch: {diff}")

print()
print("-- Final column list --")
print(result.train.columns.tolist())

-- Null check after cleaning --
  train  : OK
  val    : OK
  test   : OK

-- New columns check --
  lof_outlier     in train  : OK
  lof_outlier     in val    : OK
  lof_outlier     in test   : OK

-- lof_outlier distribution --
  train  : {0: np.int64(14208), 1: np.int64(240)} (-99=unknown, 0=inlier, 1=outlier)
  val    : {0: np.int64(3047), 1: np.int64(49)} (-99=unknown, 0=inlier, 1=outlier)
  test   : {0: np.int64(3034), 1: np.int64(62)} (-99=unknown, 0=inlier, 1=outlier)

-- Shape check --
  train  : 14,448 rows x 11 cols
  val    : 3,096 rows x 11 cols
  test   : 3,096 rows x 11 cols

-- Column consistency --
  val columns match train
  test columns match train

-- Final column list --
['longitude', 'latitude', 'housing_median_age', 'total_rooms', 'total_bedrooms', 'population', 'households', 'median_income', 'median_house_value', 'ocean_proximity', 'lof_outlier']


In [47]:
print("-- median_income: should be unchanged (NOT log1p transformed) --")
for name, df in [("train", result.train), ("val", result.val)]:
    income_max = df["median_income"].max()
    income_min = df["median_income"].min()
    # Original data range is 0.5-15.0; if unchanged, max should be > 5
    status = "OK (not transformed)" if income_max > 5 else "WARN (may have been transformed)"
    print(f"  {name:<6} : min={income_min:.2f}  max={income_max:.2f}  -> {status}")

-- median_income: should be unchanged (NOT log1p transformed) --
  train  : min=0.50  max=15.00  -> OK (not transformed)
  val    : min=0.50  max=15.00  -> OK (not transformed)


## 5 · Shape & Schema Consistency

Verify that:
- Row counts match expected split sizes (14,448 / 3,096 / 3,096)
- Column counts are identical across all three splits
- Column order is consistent (val/test match train)

In [48]:
print("-- Shape check --")
for name, df in [("train", result.train), ("val", result.val), ("test", result.test)]:
    print(f"  {name:<6} : {df.shape[0]:,} rows x {df.shape[1]} cols")

print()
print("-- Column consistency --")
train_cols = list(result.train.columns)
for name, df in [("val", result.val), ("test", result.test)]:
    if list(df.columns) == train_cols:
        print(f"  {name} columns match train")
    else:
        diff = set(train_cols) ^ set(df.columns)
        print(f"  {name} column mismatch: {diff}")

print()
print("-- Final column list --")
print(result.train.columns.tolist())

-- Shape check --
  train  : 14,448 rows x 11 cols
  val    : 3,096 rows x 11 cols
  test   : 3,096 rows x 11 cols

-- Column consistency --
  val columns match train
  test columns match train

-- Final column list --
['longitude', 'latitude', 'housing_median_age', 'total_rooms', 'total_bedrooms', 'population', 'households', 'median_income', 'median_house_value', 'ocean_proximity', 'lof_outlier']


---
## 6 - Artifacts Check

`artifacts/cleaning_artifacts.json` stores the train-fit statistics needed
to apply the exact same transformations to new production data.

In [49]:
import json
from pathlib import Path

artifacts_path = Path("artifacts/cleaning_artifacts.json")
lof_pkl_path   = Path("artifacts/lof_model.pkl")

print("-- Artifact files --")
for f in [artifacts_path, lof_pkl_path]:
    status = "OK" if f.exists() else "MISSING"
    print(f"  {status}  {f}")

if artifacts_path.exists():
    meta = json.loads(artifacts_path.read_text())
    print()
    print("-- Artifact contents --")
    print(f"  target             : {meta.get('target')}")
    print(f"  imputer_stats      : {meta.get('imputer_stats')}")
    print(f"  impute_columns     : {meta.get('impute_columns')}")
    print(f"  imputation_strategy: {meta.get('imputation_strategy')}")
    print(f"  lof_features       : {meta.get('lof_features')}")
    print(f"  lof_contamination  : {meta.get('lof_contamination')}")
    print(f"  lof_n_neighbors    : {meta.get('lof_n_neighbors')}")
    print(f"  timestamp          : {meta.get('timestamp')}")

-- Artifact files --
  OK  artifacts/cleaning_artifacts.json
  OK  artifacts/lof_model.pkl

-- Artifact contents --
  target             : median_house_value
  imputer_stats      : {'total_bedrooms': 432.0}
  impute_columns     : ['total_bedrooms']
  imputation_strategy: {'total_bedrooms': 'median'}
  lof_features       : ['median_income', 'total_rooms', 'population', 'households', 'longitude', 'latitude']
  lof_contamination  : 0.02
  lof_n_neighbors    : 20
  timestamp          : 2026-09-02T23:32:54.946642


---
## Summary & Next Steps

| Done | Details |
|---|---|
| Nulls imputed | `total_bedrooms` global median (MCAR, p=0.3095) |
| LOF outlier flag | contamination=0.02, novelty=True |
| Cleaned files saved | `data/processed/train_clean.csv` `val_clean.csv` `test_clean.csv` |
| Artifacts saved | `artifacts/cleaning_artifacts.json` + `lof_model.pkl` |

### Important Notes
- `is_capped` flag and `log1p` transformations are **not** applied in this notebook.
- These transformations belong to **feature engineering** (`05_feature_engineering.ipynb` or `dvc repro`).
- This notebook is **exploratory only** — DVC tracking is handled by the official pipeline (`dvc repro`).
